# 03. Знакомство с PyTorch — практика

Видео: [3. Знакомство с PyTorch](https://www.youtube.com/watch?v=WlwfB6bCdlw&list=PLBP4Q3FNSLK2rtGPBsK-aMAetYj-8yg_1&index=3)

Как работать:
- в ячейке с заданием замени `...` на свой код;
- запусти следующую ячейку с проверкой — если всё верно, она напечатает «верно», иначе упадёт с подсказкой;
- решения — в соседнем ноутбуке `03_знакомство_с_pytorch_решения.ipynb`, смотри только после своей попытки.

Блоки 1–7 повторяют видео. Блок 8 — разминка на формы тензоров, блок 9 — итоговая задача.

In [208]:
import torch

torch.manual_seed(0)
print(torch.__version__)


def need_tensor(x, name):
    assert isinstance(x, torch.Tensor), f"{name} — пока не тензор (замени ... на код)"

2.14.1+cpu


## 1. Создание тензоров

**Теория.** Тензор — многомерный массив чисел, как `np.array`. Матрица — частный случай:

| осей (`ndim`) | что это | пример | `shape` |
|:---|:---|:---|:---|
| 0 | скаляр | `torch.tensor(5.0)` | `()` |
| 1 | вектор | `torch.tensor([1, 2, 3])` | `(3,)` |
| 2 | матрица | `torch.tensor([[1, 2], [3, 4]])` | `(2, 2)` |
| 4 | батч картинок | 32 картинки × 3 канала × 28×28 | `(32, 3, 28, 28)` |

Отличия от NumPy: умеет записывать историю для градиентов (`requires_grad`) и считаться на видеокарте (`device`).

`torch.tensor(список)` создаёт тензор из списка. Тип PyTorch выбирает сам по числам:

```python
torch.tensor([7, 8])      # tensor([7, 8])        dtype=int64   — из целых
torch.tensor([7.0, 8])    # tensor([7., 8.])      dtype=float32 — есть хоть одно дробное
```

Дополнительные аргументы (`dtype`, `requires_grad`, `device`) нужны, только если умолчание не подходит.

**1.1** Создай одномерный тензор `a` из списка `[1, 2, 3, 4]`.

In [209]:
a = torch.tensor([1, 2, 3, 4], dtype=torch.int64)

In [210]:
need_tensor(a, "a")
assert a.shape == (4,), f"форма {tuple(a.shape)}, а нужна (4,)"
assert (
    a.dtype == torch.int64
), f"dtype {a.dtype}: из целых чисел по умолчанию получается int64"
print("1.1 — верно")

1.1 — верно


**Теория.** Вложенный список → тензор с несколькими осями: внешний список — строки, внутренние — элементы строки. Все строки должны быть **одной длины**, иначе ошибка.

`dtype` задаёт тип чисел явно. Основные:

| `dtype` | что хранит | где встречается |
|:---|:---|:---|
| `torch.float32` | дробные (по умолчанию для float) | веса, данные, loss — почти всё в нейросетях |
| `torch.float64` | дробные, двойная точность | редко: тяжелее и медленнее |
| `torch.int64` | целые (по умолчанию для int) | номера классов, индексы |
| `torch.int32` | целые, меньше памяти | иногда для экономии |
| `torch.bool` | `True` / `False` | маски |

```python
torch.tensor([[1, 0], [0, 1]], dtype=torch.float32)
# tensor([[1., 0.],
#         [0., 1.]])   — целые в списке, но тип float32: появились точки
```

**1.2** Создай двумерный тензор `b` из вложенного списка `[[1, 2, 3], [4, 5, 6]]` с типом `float32`.

In [211]:
b = torch.tensor([[1, 2, 3], [4, 5, 6]], dtype=torch.float32)

In [212]:
need_tensor(b, "b")
assert b.shape == (2, 3), f"форма {tuple(b.shape)}, а нужна (2, 3)"
assert b.dtype == torch.float32, f"dtype {b.dtype}, а нужен float32"
print("1.2 — верно")

1.2 — верно


### Как работает `requires_grad`

Флаг делает две вещи:

1. **Включает запись истории** — каждая операция с этим тензором запоминает, откуда взялся результат (`grad_fn`).
2. **Говорит «по чему» брать градиент** — после `backward()` результат ляжет в `.grad` этого тензора.

Сам флаг ничего не считает, это объявление: «следи за мной».

| что | за что отвечает | в нейросети |
|:---|:---|:---|
| `requires_grad=True` | **по чему** производная | веса `w` |
| `loss.backward()` | **от чего** производная; идёт назад по истории | от loss |
| `w.grad` | где читать результат: ∂loss/∂w | по нему обновляем `w` |

Почему по умолчанию запись **выключена**: данные огромные (батч картинок — миллионы чисел), а градиент по ним не нужен. Флаг включает запись только там, где надо — на весах.

#### Как история идёт по цепочке (на примере ниже)

Историю записывает не сам `w_demo`, а **операции**, в которых он участвует. Флаг стоит у `w_demo`, а `grad_fn` появляется у результатов:

```text
w_demo (флаг)
   ↓  pred = (w_demo * x_demo).sum()   → у pred есть grad_fn  («получился из w_demo умножением и суммой»)
   ↓  loss = (pred - y_true) ** 2      → у loss есть grad_fn  («получился из pred вычитанием и квадратом»)
```

**Правило:** если хотя бы один участник операции требует градиент, результат тоже получает `grad_fn`. У `x_demo` флага нет, но `pred` всё равно связан с историей — из-за `w_demo`. Так история «передаётся» до самого loss.

`loss.backward()` идёт по этим записям **назад** и считает по правилу цепочки:

```text
loss = (pred - y)²        →  ∂loss/∂pred = 2·(pred - y) = 2·(-2 - 1) = -6
pred = w₁·x₁ + w₂·x₂    →  ∂pred/∂w    = x = [2, 3]
                              ∂loss/∂w    = -6 · [2, 3] = [-12, -18]   → ложится в w_demo.grad
```

Чтобы пройти эту цепочку, `backward()` должен знать, **как** loss получился из `pred`, а `pred` — из `w_demo`. Это и есть записанная история.

#### Что будет без флага у `w_demo`

Флага нет ни у кого: ни у `w_demo`, ни у `x_demo`, ни у `y_true`. Значит, ни одна операция ничего не записывает:

```text
w_demo (без флага)
   ↓  pred = -2   → просто число, grad_fn нет
   ↓  loss = 9    → просто число, grad_fn нет

loss.backward()  → RuntimeError: element 0 of tensors does not require grad and does not have a grad_fn
w_demo.grad      → None
```

Важно: дело **не** в том, что нет доступа к loss — значение `9` есть. Нет связи «как `9` получилось из `w_demo`». PyTorch посчитал `-2`, потом `9` и забыл, откуда они взялись, поэтому `backward()` не по чему идти назад.

> Проверь сам: в примере ниже (пункт 3) убери `requires_grad=True` у `w_demo` и запусти ячейку. В выводе `pred` и `loss` пропадёт `grad_fn`, а на `loss.backward()` будет ошибка. Потом верни флаг.

#### Вся цепочка целиком

| шаг | что происходит | что меняется |
|:---|:---|:---|
| `requires_grad=True` | PyTorch записывает историю операций с весами | у `pred`, `loss` появляется `grad_fn` |
| `loss.backward()` | по истории назад считает ∂loss/∂w | заполняется `w.grad`; **сами веса не меняются** |
| `w -= lr * w.grad` (потом `optimizer.step()`) | двигает веса против градиента | меняется `w` |

Без первого шага не будет второго (не по чему считать градиент), а без второго — третьего (непонятно, куда двигать веса).

Запомнить:
- `backward()` **веса не меняет** — только заполняет `.grad`. Веса меняет отдельный шаг.
- `.grad` появляется только у тензоров, которым ты **сам** поставил флаг. У промежуточных (`pred`, `loss`) — `None`: градиент через них только проходит.
- `backward()` вызывают от **одного числа** — от loss.
- `.grad` **накапливается**: второй `backward()` прибавит к старому значению. Поэтому в цикле обучения перед шагом — `optimizer.zero_grad()`.
- флаг — только для float (обычно `float32`), иначе ошибка из задания 1.4.

Запусти ячейку ниже и сверь вывод с комментариями.

In [213]:
# Пример (не задание): один нейрон без bias, один пример данных

# 1. флаг выключен — истории нет
a_demo = torch.tensor([2.0])
print(a_demo * 3)  # tensor([6.])  — просто число

# 2. флаг включён — операция записана
a_demo = torch.tensor([2.0], requires_grad=True)
print(a_demo * 3)  # tensor([6.], grad_fn=<MulBackward0>)  — «получился умножением»

# 3. полноценно: веса → предсказание → loss → backward → шаг
w_demo = torch.tensor([0.5, -1.0], requires_grad=True)  # по чему: по весам
x_demo = torch.tensor([2.0, 3.0])  # данные — без флага
y_true = torch.tensor(1.0)  # правильный ответ

pred = (w_demo * x_demo).sum()  # 0.5*2 + (-1)*3 = -2
loss = (pred - y_true) ** 2  # (-2 - 1)^2 = 9 — считаем с участием w
print("pred:", pred)
print("loss:", loss)

loss.backward()  # от чего: от loss

print(
    "w_demo.grad:", w_demo.grad
)  # ∂loss/∂w = 2*(pred - y)*x = 2*(-3)*[2, 3] = [-12, -18]
print("x_demo.grad:", x_demo.grad)  # None — у данных флага нет

# 4. зачем всё это: шаг градиентного спуска (то, что потом делает optimizer.step())
lr = 0.01
with torch.no_grad():  # само обновление в историю записывать не нужно
    w_demo -= lr * w_demo.grad  # [0.5, -1.0] → [0.62, -0.82]
w_demo.grad.zero_()  # обнулить, иначе следующий backward прибавится к старому

new_loss = ((w_demo * x_demo).sum() - y_true) ** 2
print("w_demo после шага:", w_demo)
print("loss до:", loss.item(), "→ после:", round(new_loss.item(), 4))  # 9.0 → 4.9284

tensor([6.])
tensor([6.], grad_fn=<MulBackward0>)
pred: tensor(-2., grad_fn=<SumBackward0>)
loss: tensor(9., grad_fn=<PowBackward0>)
w_demo.grad: tensor([-12., -18.])
x_demo.grad: None
w_demo после шага: tensor([ 0.6200, -0.8200], requires_grad=True)
loss до: 9.0 → после: 4.9284


**Теория.** Полный разбор `requires_grad` — в ячейках выше. Коротко: `requires_grad=True` включает запись истории операций, по ней `backward()` считает градиенты. Ставится на веса, только для float.

```python
torch.tensor([1.0, 2.0], requires_grad=True)
# tensor([1., 2.], requires_grad=True)   — флаг виден в выводе
```

**1.3** Создай тензор `w` из `[0.5, -1.0, 2.0]`, для которого PyTorch будет считать градиенты.

In [214]:
w = torch.tensor([0.5, -1.0, 2.0], requires_grad=True)

In [215]:
need_tensor(w, "w")
assert w.requires_grad, "забыл requires_grad=True"
assert w.is_floating_point(), "градиенты считаются только для дробных чисел"
print("1.3 — верно")

1.3 — верно


**Теория.** Градиент — «насколько изменится результат при маленьком сдвиге числа». У целых чисел маленького сдвига нет (между 1 и 2 ничего), поэтому PyTorch запрещает `requires_grad` для `int` и кидает `RuntimeError`.

Как поймать ошибку, чтобы ячейка не упала (обычный Python):

```python
try:
    1 / 0                 # код, который может упасть
except ZeroDivisionError as e:
    print("поймали:", e)  # сюда попадаем при ошибке; e — сама ошибка, str(e) — её текст
```

**1.4** Автор показывал, что `requires_grad=True` для целых чисел даёт ошибку. Поймай её: внутри `try` создай тензор из `[1, 2, 3]` с `requires_grad=True`, а текст ошибки сохрани в `err`.

In [216]:
err = None
try:
    w = torch.tensor([1, 2, 3], requires_grad=True)
except RuntimeError as e:
    err = str(e)

print(err)

Only Tensors of floating point and complex dtype can require gradients


In [217]:
assert (
    err is not None
), "ошибка не возникла — проверь, что числа целые и стоит requires_grad=True"


assert "floating point" in err


print("1.4 — верно. Вывод: requires_grad только для float (и complex)")

1.4 — верно. Вывод: requires_grad только для float (и complex)


## 2. Свойства тензора и `item()`

**Теория.** Свойства тензора — это то, что он про себя знает. Пишутся **без скобок** (кроме `size()` и `numel()` — это методы):

| свойство | что даёт | для `torch.zeros(2, 5)` |
|:---|:---|:---|
| `.dtype` | тип чисел | `torch.float32` |
| `.shape` | форма — длина каждой оси | `torch.Size([2, 5])` |
| `.size()` | то же, что `shape` | `torch.Size([2, 5])` |
| `.ndim` | число осей = сколько чисел в `shape` | `2` |
| `.numel()` | всего элементов = произведение `shape` | `10` |
| `.device` | где лежит | `cpu` |

`torch.Size` ведёт себя как кортеж: `t.shape[0]` — длина оси 0, `t.shape == (2, 5)` — сравнение.

Как читать оси: `shape = (2, 5)` → ось 0 длины 2 (строки), ось 1 длины 5 (столбцы). Число скобок `[[` в начале вывода = `ndim`.

**2.1** Дан тензор `x`. Запиши в переменные его тип, форму и число измерений, используя свойства тензора (не вписывай числа руками).

In [218]:
x = torch.rand(2, 3, 4)

In [219]:
x_dtype = x.dtype
print(x_dtype)
x_shape = x.shape
print(x_shape)
x_ndim = x.ndim
print(x_ndim)
print(x)

torch.float32
torch.Size([2, 3, 4])
3
tensor([[[0.4963, 0.7682, 0.0885, 0.1320],
         [0.3074, 0.6341, 0.4901, 0.8964],
         [0.4556, 0.6323, 0.3489, 0.4017]],

        [[0.0223, 0.1689, 0.2939, 0.5185],
         [0.6977, 0.8000, 0.1610, 0.2823],
         [0.6816, 0.9152, 0.3971, 0.8742]]])


In [220]:
assert x_dtype == torch.float32
assert x_shape == (2, 3, 4) and x_shape == x.size(), "shape и size() дают одно и то же"
assert x_ndim == 3
print("2.1 — верно")

2.1 — верно


**Теория.** Индексация как в NumPy: по одному индексу на каждую ось, через запятую. Нумерация с 0, отрицательные — с конца.

```python
t = torch.tensor([[10, 20, 30],
                  [40, 50, 60]])

t[0, 2]     # tensor(30)          строка 0, столбец 2
t[-1, -1]   # tensor(60)          последняя строка, последний столбец
t[1]        # tensor([40, 50, 60])  только ось 0 → вся строка, форма (3,)
t[:, 1]     # tensor([20, 50])    все строки, столбец 1, форма (2,)
```

Даже один элемент — это **тензор** без осей (`shape = ()`), а не число. Обычное Python-число — через `.item()`; работает только если элемент ровно один:

```python
t[0, 2].item()   # 30  (int)
t[1].item()      # ❌ ошибка: там 3 элемента
```

**2.2** Из тензора `m` достань элемент `5`:
- `el` — как тензор (через индексацию);
- `num` — как обычное число Python.

In [221]:
m = torch.tensor([[1.0, 2.0, 3.0], [4.0, 5.0, 6.0]], requires_grad=True)

In [222]:
el = m[1, 1]
num = m[1, 1].item()

print(el, type(el))
print(num, type(num))

tensor(5., grad_fn=<SelectBackward0>) <class 'torch.Tensor'>
5.0 <class 'float'>


In [223]:
need_tensor(el, "el")
assert el.ndim == 0, "индексация одного элемента даёт тензор без измерений (скаляр)"
assert isinstance(num, float) and num == 5.0, "число получают методом item()"
print("2.2 — верно")

2.2 — верно


**Теория.** В цикле обучения loss каждой эпохи сохраняют в список, чтобы потом нарисовать график. Сохраняют через `.item()` — обычные числа: не тянут за собой историю операций и не держат память.

Собрать список из другого списка — обычный Python (list comprehension):

```python
squares = [n * n for n in [1, 2, 3]]   # [1, 4, 9]
```

**2.3** Так `item()` используют на практике: сохраняют значение loss по эпохам. Собери в список `history` значения из `losses` как обычные числа.

> `losses` — **условные** loss за 3 эпохи, не настоящая формула. `(w * k).sum()` нужен только чтобы получить одно число с `grad_fn`, как у настоящего loss (`w` — из задания 1.3).
>
> Как бывает на деле: внутри батча loss **усредняет** ошибки по примерам (MSE = `((pred - y) ** 2).mean()`), поэтому это одно число. За эпоху для графика берут среднее по батчам. Между эпохами ничего не складывают — каждое значение записывают в список по порядку: `history = [0.92, 0.61, 0.45, ...]`.

In [224]:
losses = [(w * k).sum() for k in range(1, 4)]
print(losses)

[tensor(1.5000, grad_fn=<SumBackward0>), tensor(3., grad_fn=<SumBackward0>), tensor(4.5000, grad_fn=<SumBackward0>)]


In [225]:
history = [loss.item() for loss in losses]

In [226]:
assert isinstance(history, list) and len(history) == 3
assert all(
    isinstance(v, float) for v in history
), "в history должны быть float, а не тензоры"
print("2.3 — верно:", history)

2.3 — верно: [1.5, 3.0, 4.5]


> Вопрос себе: почему в истории лучше хранить числа, а не сами тензоры `losses`? Подсказка: что ещё тащит за собой тензор (смотри `grad_fn` в выводе выше).

### Почему PyTorch возвращает тензор, а не сразу число

Индексация `m[1, 1]`, `x.sum()`, loss — всегда тензор, даже если внутри одно число (тензор без осей, `shape = ()`). Тензор нужен **для дальнейших вычислений**, обычное число — только чтобы **вывести или сохранить**. Если бы сразу возвращалось число, сломались бы три вещи:

| что сломалось бы | почему |
|:---|:---|
| **градиенты** | у Python-числа нет `grad_fn` и `backward()` — оно не помнит, что получилось из весов. Будь loss сразу `float`, обучить сеть было бы нельзя |
| **скорость на GPU** | чтобы получить число с видеокарты, надо дождаться, пока она досчитает, и скопировать результат в обычную память. Это дорого; делай так каждая операция — GPU бы простаивал. С `item()` ты сам решаешь когда: обычно раз за батч или эпоху |
| **тип и устройство** | тензор помнит `dtype` и `device`, его можно сразу пустить дальше в вычисления. Python-число этого не знает |

Правило PyTorch: **операции с тензорами всегда возвращают тензоры**, а выход в обычный Python — явный, через `item()`. В NumPy похоже: `np.array([1.5, 2.0])[0]` — это `np.float64`, а не обычный `float`.

### Градиенты работают только с тензорами

Причём **вся цепочка от весов до loss** должна оставаться тензорами и операциями PyTorch. Вышел посередине в Python или NumPy — история обрывается:

```python
pred = (w_demo * x_demo).sum()

loss = (pred - y_true) ** 2              # ✅ всё тензоры → backward работает

p = pred.item()                          # вышли в обычное число
loss = (torch.tensor(p) - y_true) ** 2   # ❌ связи с w_demo уже нет → backward упадёт
```

С NumPy то же: `.numpy()` → посчитал там → обратно в тензор — история потеряна. PyTorch даже не даст вызвать `.numpy()` у тензора с флагом без `detach()` — чтобы цепочку не порвали незаметно.

**Правило:** пока считаешь loss — только операции PyTorch. `item()`, `.numpy()`, `print` — только после, для вывода и сохранения.

## 3. Функции создания

`zeros`, `ones`, `zeros_like`, `full`, `full_like`, `arange`, `eye`, `diag`, `tril`.

**Теория.** Чтобы не писать списки руками, тензор создают готовыми функциями — сразу нужной формы:

```python
torch.zeros(2, 3)                    # нули, форма (2, 3), dtype float32
torch.ones(4)                        # единицы, форма (4,)
torch.zeros(2, 3, dtype=torch.int64) # тип можно задать, как в torch.tensor
```

Форма передаётся числами через запятую (`zeros(2, 3)`) или кортежем (`zeros((2, 3))`) — одно и то же. По умолчанию тип **float32**.

Версии с `_like` берут у образца **форму, тип и устройство**:

```python
sample = torch.tensor([[1.0, 2.0, 3.0]])   # (1, 3), float32
torch.zeros_like(sample)                   # tensor([[0., 0., 0.]])  — та же форма (1, 3)
torch.ones_like(sample)                    # tensor([[1., 1., 1.]])
```

Зачем: форма заранее неизвестна или может меняться (последний батч эпохи бывает меньше) — `_like` подстроится сам, и не надо помнить, на CPU образец или на GPU.

**3.1** Создай:
- `z` — нули формы (3, 4);
- `zi` — нули формы (3, 4) с типом `int32`;
- `o` — единицы формы (2, 2);
- `zl` — нули той же формы, что у `b` (не вписывай форму руками).

In [227]:
z = torch.zeros(3, 4)
zi = torch.zeros(3, 4, dtype=torch.int32)
o = torch.ones(2, 2)
zl = torch.zeros_like(b)

In [228]:
for t, name in [(z, "z"), (zi, "zi"), (o, "o"), (zl, "zl")]:
    need_tensor(t, name)
assert z.shape == (3, 4) and z.sum() == 0 and z.dtype == torch.float32
assert zi.dtype == torch.int32
assert o.shape == (2, 2) and o.sum() == 4
assert zl.shape == b.shape and zl.sum() == 0
print("3.1 — верно")

3.1 — верно


**Теория.** `full` — как `zeros`/`ones`, только заполняет **любым** числом:

| | форма | тип |
|:---|:---|:---|
| `torch.full((2, 2), 3.5)` | задаёшь сам (кортежем) | по числу: `3.5` → float32 |
| `torch.full_like(sample, 3.5)` | берётся у `sample` | тоже у `sample`! |

Ловушка `full_like`: если образец целочисленный, дробное число обрежется:

```python
ints = torch.tensor([[1, 2, 3], [4, 5, 6]])   # int64
torch.full_like(ints, 7.5)   # tensor([[7, 7, 7], [7, 7, 7]])  — 7.5 стало 7
torch.full((2, 3), 7.5)      # tensor([[7.5, 7.5, 7.5], ...])  — float
```

**3.2** Тензоры, заполненные одним числом:
- `f1` — форма (2, 3), все элементы `7.0` (`torch.full`, форму задаёшь сам);
- `f2` — форма как у `b`, все элементы `7.0` (`torch.full_like`, форму берёт из тензора).

In [229]:
f1 = torch.full((2, 3), 7.0)
f2 = torch.full_like(b, 7.0)

In [230]:
need_tensor(f1, "f1")
need_tensor(f2, "f2")


assert f1.shape == (2, 3) and (f1 == 7).all()


assert f2.shape == b.shape and (f2 == 7).all()


print("3.2 — верно")

3.2 — верно


**Теория.**

`torch.arange(start, end, step)` — числа от `start` до `end` с шагом `step`, **`end` не входит** (как `range` и `np.arange`):

```python
torch.arange(5)            # tensor([0, 1, 2, 3, 4])
torch.arange(1, 10, 3)     # tensor([1, 4, 7])           — 10 не входит
torch.arange(0, 1, 0.25)   # tensor([0.00, 0.25, 0.50, 0.75])  — шаг может быть дробным
```

`torch.eye(n)` — единичная матрица n×n: единицы на диагонали, остальное нули. Умножение на неё ничего не меняет (`A @ eye = A`). Её строки — готовые one-hot векторы:

```python
torch.eye(4)[torch.tensor([3, 0])]
# tensor([[0., 0., 0., 1.],    ← класс 3
#         [1., 0., 0., 0.]])   ← класс 0
```

`torch.diag(вектор)` — матрица с этим вектором на диагонали (и наоборот: `torch.diag(матрица)` достаёт её диагональ):

```python
torch.diag(torch.tensor([7, 8]))   # tensor([[7, 0],
                                   #         [0, 8]])
```

`torch.tril(матрица)` — оставляет **нижний треугольник** с диагональю (элемент `[i, j]` остаётся, если `i >= j`), остальное обнуляет. `torch.triu` — верхний:

```python
m = torch.arange(1, 10).view(3, 3)   # [[1, 2, 3], [4, 5, 6], [7, 8, 9]]
torch.tril(m)                        # [[1, 0, 0],
                                     #  [4, 5, 0],
                                     #  [7, 8, 9]]
```

Где встречается: `tril` от единиц — маска «каждая позиция видит только себя и предыдущие» в трансформерах для генерации текста.

**3.3**
- `r` — чётные числа от 0 до 8 включительно через `torch.arange`;
- `e` — единичная матрица 3×3;
- `d` — диагональная матрица с `1, 2, 3` на диагонали;
- `lt` — нижнетреугольная матрица 3×3 из единиц (подсказка: `torch.tril` от матрицы единиц).

In [231]:
r = torch.arange(0, 9, 2)
e = torch.eye(3)
d = torch.diag(torch.tensor([1, 2, 3]))
lt = torch.tril(torch.ones(3, 3))

In [232]:
for t, name in [(r, "r"), (e, "e"), (d, "d"), (lt, "lt")]:
    need_tensor(t, name)
assert r.tolist() == [
    0,
    2,
    4,
    6,
    8,
], f"получилось {r.tolist()}: правая граница в arange не входит"
assert torch.equal(e, torch.tensor([[1.0, 0, 0], [0, 1, 0], [0, 0, 1]]))
assert d.tolist() == [[1, 0, 0], [0, 2, 0], [0, 0, 3]]
assert lt.tolist() == [[1, 0, 0], [1, 1, 0], [1, 1, 1]]
print("3.3 — верно")
print(lt)

3.3 — верно
tensor([[1., 0., 0.],
        [1., 1., 0.],
        [1., 1., 1.]])


## 4. Изменение формы: `view`, `reshape`, `unsqueeze`

**Теория.** `view` и `reshape` **перекладывают те же числа в новую форму**. Сами числа и их порядок не меняются — меняется только, как они разбиты на оси. Раскладка идёт по строкам, слева направо:

```python
t = torch.arange(6)        # tensor([0, 1, 2, 3, 4, 5])   (6,)
t.view(2, 3)               # [[0, 1, 2],
                           #  [3, 4, 5]]                    (2, 3)
t.reshape(3, 2)            # [[0, 1],
                           #  [2, 3],
                           #  [4, 5]]                       (3, 2)
```

Правило: **произведение новой формы = числу элементов**. 6 = 2·3 = 3·2 ✅, а `t.view(4, 2)` — 8 ≠ 6 → ошибка.

`-1` вместо одного размера — «посчитай сам»: `t.view(2, -1)` → 6 / 2 = 3 → `(2, 3)`. `-1` можно поставить только в **одно** место.

Чем отличаются `view` и `reshape` — в задании 4.3. Пока считай, что одинаково.

**4.1** Дан `v = torch.arange(12)`. Получи:
- `v34` — форма (3, 4) через `view`;
- `v223` — форма (2, 2, 3) через `reshape`;
- `v4x` — 4 строки, а число столбцов пусть PyTorch посчитает сам (через `-1`).

In [233]:
v = torch.arange(12)

In [234]:
v34 = v.view([3, 4])
v223 = v.reshape([2, 2, 3])
v4x = v.reshape([4, -1])

In [235]:
for t, name in [(v34, "v34"), (v223, "v223"), (v4x, "v4x")]:
    need_tensor(t, name)
assert v34.shape == (3, 4)
assert v223.shape == (2, 2, 3)
assert v4x.shape == (4, 3), "-1 значит: вычисли этот размер сам (12 / 4 = 3)"
print("4.1 — верно")
print(v34)

4.1 — верно
tensor([[ 0,  1,  2,  3],
        [ 4,  5,  6,  7],
        [ 8,  9, 10, 11]])


**Теория.** `unsqueeze(dim)` **вставляет новую ось размера 1** на место `dim`. Чисел не добавляет — только «обёртку». `squeeze(dim)` — обратно: убирает ось, но **только если её размер 1**.

```python
t = torch.tensor([7, 8, 9, 10])   # (4,)
t.unsqueeze(0)                    # [[7, 8, 9, 10]]          (1, 4)  — ось слева
t.unsqueeze(1)                    # [[7], [8], [9], [10]]    (4, 1)  — ось справа

t.unsqueeze(0).squeeze(0)         # (4,)  — убрали обратно
torch.zeros(2, 4).squeeze(0)      # (2, 4) — ось 0 размера 2 не убирается, и ошибки нет
```

Зачем: модели всегда ждут **батч** первой осью. Одна картинка `(3, 28, 28)` → `unsqueeze(0)` → `(1, 3, 28, 28)` — «батч из одной».

Совет: в `squeeze` всегда указывай ось. Без аргумента `squeeze()` убирает **все** оси размера 1 — в батче из одного примера пропадёт и ось батча.

**4.2** `unsqueeze(dim)` добавляет ось размера 1. Дан `p` формы (3,). Получи:
- `p_row` — форма (1, 3);
- `p_col` — форма (3, 1);
- `p_back` — из `p_row` обратно форма (3,) через `squeeze`.

In [236]:
p = torch.tensor([1.0, 2.0, 3.0])
print(p.shape)
asd = p.unsqueeze(0)

torch.Size([3])


In [237]:
p_row = p.unsqueeze(0)
p_col = p.unsqueeze(1)
p_back = p.squeeze(0)

In [238]:
for t, name in [(p_row, "p_row"), (p_col, "p_col"), (p_back, "p_back")]:
    need_tensor(t, name)
assert p_row.shape == (1, 3)
assert p_col.shape == (3, 1)
assert p_back.shape == (3,)
print("4.2 — верно")

4.2 — верно


**Теория.** В памяти числа тензора лежат одной длинной строкой. Форма — это лишь «как читать» эту строку.

- `.T` (транспонирование) **не переставляет числа в памяти** — он только меняет правило чтения. Данные теперь «вразнобой» относительно новой формы (тензор *не непрерывный*, `t.is_contiguous()` → `False`).
- `view` умеет только перечитать память по-новому, **без копирования** → на таком тензоре падает.
- `reshape` сначала пробует как `view`, а если нельзя — **копирует** данные в нужном порядке. Поэтому работает всегда.

```python
t = torch.arange(4).view(2, 2).T   # [[0, 2],
                                   #  [1, 3]]
t.is_contiguous()                  # False
t.reshape(4)                       # tensor([0, 2, 1, 3])   ✅
```

Практическое правило: не уверен — пиши `reshape`.

**4.3** Автор сказал, что `view` и `reshape` делают одно и то же. Почти: `view` работает только когда данные лежат в памяти подряд. После транспонирования (`.T`) это не так.

Дан `mt`. Внутри `try` попробуй вытянуть его в вектор через `view(6)`, текст ошибки сохрани в `err_view`. Затем получи вектор `flat` через `reshape`.

In [239]:
mt = torch.arange(6).view(2, 3).T
print(mt)

tensor([[0, 3],
        [1, 4],
        [2, 5]])


In [240]:
err_view = None
try:
    err_view = mt.view(6)
except RuntimeError as e:
    err_view = str(e)

flat = mt.reshape(6)

In [241]:
assert err_view is not None and "view size is not compatible" in err_view
need_tensor(flat, "flat")
assert flat.tolist() == [0, 3, 1, 4, 2, 5]
print("4.3 — верно. Правило: если не уверен — reshape")

4.3 — верно. Правило: если не уверен — reshape


## 5. Арифметика

**Теория.** Обычные операторы `+ - * / **` работают **поэлементно**:

```python
t = torch.tensor([1.0, 2.0, 4.0])

t + 10        # tensor([11., 12., 14.])  — число применяется к каждому элементу
t ** 2        # tensor([ 1.,  4., 16.])
t * t         # tensor([ 1.,  4., 16.])  — тензор той же формы: элемент с элементом
```

Важно: `*` — **не** матричное умножение. Матричное — `@` (или `torch.matmul`):

```python
A = torch.tensor([[1., 2.], [3., 4.]])
A * A         # [[1., 4.], [9., 16.]]    — поэлементно
A @ A         # [[7., 10.], [15., 22.]]  — строка на столбец
```

**5.1** Дан `q`. Получи `q2` — `q`, умноженный на 2, и `qq` — сумму `q` с самим собой.

In [242]:
q = torch.tensor([1.0, 2.0, 3.0])

In [243]:
q2 = q * 2
qq = q + q

In [244]:
need_tensor(q2, "q2")
need_tensor(qq, "qq")


assert q2.tolist() == [2.0, 4.0, 6.0] and torch.equal(q2, qq)


print("5.1 — верно")

5.1 — верно


**Теория.** Тензор умеет считать с числами и с другими тензорами, но **не со списками** Python. Список сначала превращают в тензор — `torch.tensor(список)`.

Узнать тип пойманной ошибки — `type(e).__name__`:

```python
try:
    int("abc")
except Exception as e:
    print(type(e).__name__)   # ValueError
```

`except Exception` ловит почти любую ошибку — удобно, когда заранее не знаешь её тип.

**5.2** Сложение тензора со списком падает. Поймай тип ошибки в `err_type` (например `"ValueError"`), а потом исправь: получи `fixed` = `q` + `[10, 20, 30]`, превратив список в тензор.

In [245]:
err_type = None
try:
    q + [10, 20, 30]
except Exception as e:
    err_type = type(e).__name__

fixed = torch.tensor([10, 20, 30]) + q

In [246]:
assert err_type == "TypeError", f"получил {err_type}"
need_tensor(fixed, "fixed")
assert fixed.tolist() == [11.0, 22.0, 33.0]
print("5.2 — верно")

5.2 — верно


## 6. `sum` / `mean` по осям и `keepdim`

`dim` — по какой оси сворачиваем. Эта ось из формы **исчезает**, а с `keepdim=True` остаётся с размером 1.

**Теория.** `sum` / `mean` без аргументов сворачивают **весь** тензор в одно число. С `dim=k` — идут **вдоль оси k** и сворачивают только её, остальные оси остаются.

```python
t = torch.tensor([[1., 2., 3.],
                  [4., 5., 6.]])    # (2, 3): ось 0 — строки, ось 1 — столбцы

t.sum()          # tensor(21.)                всё в одно число
t.sum(dim=0)     # tensor([5., 7., 9.])       вдоль оси 0: 1+4, 2+5, 3+6  → (3,)
t.sum(dim=1)     # tensor([ 6., 15.])         вдоль оси 1: 1+2+3, 4+5+6 → (2,)
t.sum(dim=1, keepdim=True)   # [[ 6.], [15.]] ось не исчезла, стала 1  → (2, 1)
```

**Правило формы** без вычислений: вычеркни из `shape` ось `dim` → `(2, 3)` при `dim=1` → убираем ось 1 (длина 3) → `(2,)`. С `keepdim=True` вместо вычёркивания ставь 1 → `(2, 1)`.

Ещё:
- `dim=-1` — последняя ось (здесь то же, что `dim=1`);
- можно сразу несколько осей кортежем: `dim=(0, 1)`;
- `mean` работает **только с float**: `torch.arange(6).mean()` → ошибка (int64). Тогда `torch.arange(6, dtype=torch.float32)` или `.float()`.

#### `keepdim` не добавляет ось, а не даёт её удалить

`sum` / `mean` по оси `dim` эту ось выкидывают. С `keepdim=True` она **остаётся на своём месте**, только длиной 1. Поэтому число осей с флагом — **как у исходного тензора**.

На `s = torch.arange(6, dtype=torch.float32).view(2, 3)` → `[[0, 1, 2], [3, 4, 5]]`:

| код | результат | форма | осей |
|:---|:---|:---|:---|
| `s` | `[[0, 1, 2], [3, 4, 5]]` | `(2, 3)` | 2 |
| `s.sum(dim=1)` | `[3., 12.]` | `(2,)` | **1** — ось 1 удалена |
| `s.sum(dim=1, keepdim=True)` | `[[3.], [12.]]` | `(2, 1)` | **2** — ось 1 на месте, длина 3 → 1 |
| `s.sum(dim=0)` | `[3., 5., 7.]` | `(3,)` | **1** — ось 0 удалена |
| `s.sum(dim=0, keepdim=True)` | `[[3., 5., 7.]]` | `(1, 3)` | **2** — ось 0 на месте, длина 2 → 1 |

```text
(2, 3)   dim=1                →  (2,)     ← ось 1 удалена
(2, 3)   dim=1, keepdim=True  →  (2, 1)   ← ось 1 на месте, но длина 3 → 1
(2, 3)   dim=0                →  (3,)
(2, 3)   dim=0, keepdim=True  →  (1, 3)
```

Чтобы не путаться в словах:
- `dim=1` (сворачиваем ось **столбцов**) → складываются числа внутри каждой строки → **одно число на строку**: `[3, 12]`;
- `dim=0` (сворачиваем ось **строк**) → складываются числа внутри каждого столбца → **одно число на столбец**: `[3, 5, 7]`.

Проще всего думать так: **ось `dim` исчезает (или с флагом становится 1), остальные оси остаются как были**.

Разница с `unsqueeze`: он **добавляет** новую ось (было 1 ось → стало 2). Результат одинаковый, путь разный:

```python
s.sum(dim=1, keepdim=True)   # (2, 1) — ось не удалили
s.sum(dim=1).unsqueeze(1)    # (2, 1) — удалили и вставили обратно
```

#### Зачем нужен `keepdim` — для broadcasting

Чтобы результат свёртки можно было сразу вернуть в операцию с **исходным** тензором (вычесть, поделить, сложить). Ось длины 1 остаётся на своём месте → broadcasting растягивает её туда, куда нужно (подробно — в теории к заданию 6.2).

```python
x - x.mean(dim=1, keepdim=True)          # вычесть среднее каждой строки
x / x.sum(dim=1, keepdim=True)           # в каждой строке сумма станет 1 (как вероятности)
X - X.mean(dim=(1, 2, 3), keepdim=True)  # вычесть среднее каждой картинки батча (задача 9)
```

- Результат дальше никуда не возвращается (вывести, сохранить, loss) → `keepdim` не нужен.
- Для 2D при `dim=0` сработает и без флага (`(3,)` и так встаёт напротив столбцов), но на это лучше не полагаться: с `keepdim=True` работает всегда, при любой оси.

In [247]:
s = torch.arange(6, dtype=torch.float32).view(2, 3)

**6.1** Посчитай:
- `s_sum` — сумма всех элементов;
- `s_mean` — среднее всех элементов;
- `col_mean` — среднее каждого столбца (форма (3,));
- `row_mean` — среднее каждой строки (форма (2,));
- `row_mean_k` — среднее каждой строки с `keepdim=True`.

Перед запуском проверки предскажи, какие получатся числа.

In [248]:
s_sum = s.sum()
s_mean = s.mean()
col_mean = s.mean(dim=0)
row_mean = s.mean(dim=1)
row_mean_k = s.mean(dim=1, keepdim=True)

In [249]:
for t, name in [
    (s_sum, "s_sum"),
    (s_mean, "s_mean"),
    (col_mean, "col_mean"),
    (row_mean, "row_mean"),
    (row_mean_k, "row_mean_k"),
]:

    need_tensor(t, name)


assert s_sum.item() == 15.0 and s_mean.item() == 2.5


assert col_mean.tolist() == [
    1.5,
    2.5,
    3.5,
], "dim=0 — сворачиваем строки, остаётся по числу на столбец"


assert row_mean.tolist() == [1.0, 4.0]


assert row_mean_k.shape == (
    2,
    1,
), f"форма {tuple(row_mean_k.shape)}, а с keepdim нужна (2, 1)"


print("6.1 — верно")

6.1 — верно


### Теория: broadcasting

Когда в поэлементной операции (`+ - * /`) у тензоров разные формы, PyTorch (как NumPy) пробует **«растянуть»** меньший — повторить его, пока формы не совпадут. Размер оси **1** значит: «тут одно значение — используй его везде вдоль этой оси».

> Не путать с матричным умножением `@`: там столбцы первой сравниваются со строками второй (крест-накрест). В broadcasting сравниваются **одинаковые** оси: последняя с последней, предпоследняя с предпоследней.

#### Правило — 3 шага

1. **Выровнять формы по правому краю.**
2. **Где у короткой формы пусто — дописать 1 слева.** `(3,)` → `(1, 3)`: та же строка, просто в «обёртке».
3. **Сравнить оси попарно, справа налево.** В каждой паре размеры либо **равны**, либо один **равен 1** (он растягивается до второго). Иначе — ошибка.

#### Пример 1: `(2, 3)` + `(3,)` — строка прибавляется к каждой строке

```python
s = torch.tensor([[0, 1, 2],
                  [3, 4, 5]])     # (2, 3)
s + torch.tensor([10, 20, 30])    # (3,)
```

```text
шаг 1:  s:      (2, 3)
        вектор:    (3,)

шаг 2:  s:      (2, 3)
        вектор: (1, 3)        ← дописали 1 слева

шаг 3:  последняя:      3 и 3  → равны ✅
        предпоследняя:  2 и 1  → 1 растягивается до 2 ✅
        итог:           (2, 3)
```

«Растягивается до 2» = строка `[10, 20, 30]` используется для обеих строк:

```text
[[0, 1, 2],     +   [[10, 20, 30],     =   [[10, 21, 32],
 [3, 4, 5]]          [10, 20, 30]]          [13, 24, 35]]
                     ↑ та же строка, повторённая
```

#### Пример 2: `(2, 3)` + `(2, 1)` — столбец прибавляется к каждому столбцу

```python
s + torch.tensor([[100],
                  [200]])         # (2, 1)
```

```text
шаг 1:  s:       (2, 3)
        столбец: (2, 1)       ← выровняли по правому краю

шаг 2:  осей у обоих по 2 — дописывать ничего не надо

шаг 3:  последняя:      3 и 1  → 1 растягивается до 3 ✅
        предпоследняя:  2 и 2  → равны ✅
        итог:           (2, 3)
```

```text
[[0, 1, 2],     +   [[100, 100, 100],     =   [[100, 101, 102],
 [3, 4, 5]]          [200, 200, 200]]          [203, 204, 205]]
                     ↑ столбец, повторённый 3 раза вправо
```

#### Пример 3: `(2, 3)` + `(2,)` — ошибка

```python
s + torch.tensor([100, 200])      # (2,)
```

```text
шаг 1:  s:      (2, 3)
        вектор:    (2,)        ← выровняли по правому краю: 2 оказалось напротив оси длины 3

шаг 2:  s:      (2, 3)
        вектор: (1, 2)        ← дописали 1 слева

шаг 3:  последняя:      3 и 2  → не равны и ни одна не 1 ❌ → ошибка
        (дальше не сравниваем — достаточно одной неподходящей пары)
```

PyTorch не догадывается, что `(2,)` значит «по числу на каждую **строку**» — он всегда ставит короткую форму по правому краю.

#### Отсюда смысл `keepdim=True`

Среднее по строкам (`mean(dim=1)`) — это «по одному числу на строку»:

| | форма | `s - среднее` |
|:---|:---|:---|
| без `keepdim` | `(2,)` | как пример 3 → ошибка |
| с `keepdim=True` | `(2, 1)` | как пример 2 → среднее строки вычитается из каждого её элемента ✅ |

Ось размера 1, оставленная `keepdim`, сама показывает PyTorch, вдоль чего растягивать.

**6.2** Зачем нужен `keepdim`. Хотим из каждой строки `s` вычесть её среднее, чтобы среднее строки стало 0.

- внутри `try` вычти `row_mean` (форма (2,)) из `s` (форма (2, 3)), текст ошибки сохрани в `err_bc`;
- потом получи `centered` правильно, через `row_mean_k`.

In [250]:
err_bc = None
try:
    err_bc = s - row_mean
except RuntimeError as e:

    err_bc = str(e)


print(err_bc)


centered = s - row_mean_k

The size of tensor a (3) must match the size of tensor b (2) at non-singleton dimension 1


In [251]:
assert err_bc is not None, "ошибки не было — вычитать нужно row_mean без keepdim"
need_tensor(centered, "centered")
assert centered.shape == (2, 3)
assert torch.allclose(centered.mean(dim=1), torch.zeros(2))
print("6.2 — верно")
print(centered)

6.2 — верно
tensor([[-1.,  0.,  1.],
        [-1.,  0.,  1.]])


> Почему так: (2, 3) и (2,) PyTorch выравнивает **справа** — 3 против 2, не совпадает. А (2, 3) и (2, 1) — ось размера 1 растягивается до 3, всё работает. Это broadcasting, как в NumPy.

### Разбор: `dim` и `keepdim` (для повторения)

Это **два разных аргумента**, они не заменяют друг друга:

| аргумент | на какой вопрос отвечает | что туда пишут |
|:---|:---|:---|
| `dim` | **по какой оси** сворачивать | номер оси: `0`, `1`, `-1`, `(1, 2)` |
| `keepdim` | **оставить ли эту ось** в форме (размером 1) | только `True` / `False` |

- `dim` — главный. `keepdim` — необязательная добавка **к** `dim`, по умолчанию `False`.
- `keepdim` **без** `dim` не работает: ось не указана — оставлять нечего.
- Числа с `keepdim` и без него **одинаковые** — отличается только форма.

**Когда нужен `keepdim=True`:** когда результат потом вычитаешь из исходного тензора / делишь / складываешь с ним. Форма с осью размера 1 правильно растягивается (broadcasting). Во всех остальных случаях хватает просто `dim`.

Ниже три ячейки — запусти и сверь вывод с комментариями:
1. что дают `mean()`, `mean(dim=1)`, `mean(dim=1, keepdim=True)`;
2. зачем `keepdim`: вычесть из каждой строки её среднее;
3. частые ошибки и как их исправить (в т.ч. что делать, если `(2,)` дан готовым и `keepdim` применить не к чему).

In [252]:
# 1. что дают dim и keepdim — числа одинаковые, отличается только форма
s_demo = torch.tensor([[0.0, 1.0, 2.0], [3.0, 4.0, 5.0]])  # (2, 3)

all_mean = s_demo.mean()  # свернули ВСЁ в одно число
row_mean_demo = s_demo.mean(dim=1)  # свернули ось 1 (столбцы) → она исчезла
row_mean_demo_k = s_demo.mean(
    dim=1, keepdim=True
)  # свернули ось 1, но ОСТАВИЛИ её размером 1

print("mean():", all_mean, all_mean.shape)
# tensor(2.5000) torch.Size([])            — (0+1+2+3+4+5) / 6

print("mean(dim=1):", row_mean_demo, row_mean_demo.shape)
# tensor([1., 4.]) torch.Size([2])         — два числа подряд: (0+1+2)/3 = 1, (3+4+5)/3 = 4

print("mean(dim=1, keepdim=True):")
print(row_mean_demo_k, row_mean_demo_k.shape)
# tensor([[1.],
#         [4.]]) torch.Size([2, 1])        — те же два числа, но столбиком: «по числу на строку»

mean(): tensor(2.5000) torch.Size([])
mean(dim=1): tensor([1., 4.]) torch.Size([2])
mean(dim=1, keepdim=True):
tensor([[1.],
        [4.]]) torch.Size([2, 1])


In [253]:
# 2. зачем keepdim: вычесть из каждой строки её среднее (чтобы среднее строки стало 0)

# без keepdim: (2, 3) - (2,) → (2,) встаёт напротив столбцов (их 3) → 3 и 2 не совпадают → ошибка
try:
    s_demo - s_demo.mean(dim=1)
except RuntimeError as e:
    print("без keepdim → ошибка:", e)
# The size of tensor a (3) must match the size of tensor b (2) at non-singleton dimension 1

# с keepdim: (2, 3) - (2, 1) → столбец [[1], [4]] растягивается вправо до (2, 3)
centered_demo = s_demo - s_demo.mean(dim=1, keepdim=True)
print("с keepdim:")
print(centered_demo)
# tensor([[-1., 0., 1.],     ← из первой строки вычли 1
#         [-1., 0., 1.]])    ← из второй вычли 4
#
# как посчиталось:
# [[0, 1, 2],      [[1, 1, 1],       [[-1, 0, 1],
#  [3, 4, 5]]  −    [4, 4, 4]]   =    [-1, 0, 1]]
#                   ↑ столбец [[1], [4]], растянутый вправо

print("среднее каждой строки теперь:", centered_demo.mean(dim=1))  # tensor([0., 0.])

без keepdim → ошибка: The size of tensor a (3) must match the size of tensor b (2) at non-singleton dimension 1
с keepdim:
tensor([[-1.,  0.,  1.],
        [-1.,  0.,  1.]])
среднее каждой строки теперь: tensor([0., 0.])


In [254]:
# 3. частые ошибки с dim / keepdim и как правильно
v_demo = torch.tensor([10, 20])  # int64 — целые числа

# ❌ keepdim без dim (и keepdim — это True/False, а не номер оси)
try:
    v_demo.float().mean(keepdim=0)
except TypeError as e:
    print("keepdim без dim → TypeError:", str(e)[:60], "...")

# ❌ mean у целых чисел: среднее может быть дробным, а int64 его не хранит
try:
    v_demo.mean(dim=0)
except RuntimeError as e:
    print("mean у int → RuntimeError:", str(e)[:60], "...")

# ✅ правильно
print(
    v_demo.float().mean(dim=0)
)  # tensor(15.)    — сначала в float, потом среднее по оси 0
print(
    v_demo.float().mean(dim=0, keepdim=True)
)  # tensor([15.])  — ось 0 осталась, форма (1,)
print(
    v_demo.mean(dtype=torch.float32)
)  # tensor(15.)    — или попросить float прямо в mean

# если (2,) дан ГОТОВЫМ (не из mean/sum) — keepdim применить не к чему.
# тогда ось размера 1 добавляют вручную: unsqueeze(1) / view(2, 1) / reshape(2, 1)
s_int = torch.tensor([[0, 1, 2], [3, 4, 5]])  # (2, 3)
print(s_int + v_demo.unsqueeze(1))  # (2, 3) + (2, 1) → (2, 3)
# tensor([[10, 11, 12],     ← 10 прибавилось ко всей первой строке
#         [23, 24, 25]])    ← 20 — ко всей второй
# а просто s_int + v_demo упадёт — как пример 3 в теории broadcasting

keepdim без dim → TypeError: mean() received an invalid combination of arguments - got (k ...
mean у int → RuntimeError: mean(): could not infer output dtype. Input dtype must be ei ...
tensor(15.)
tensor([15.])
tensor(15.)
tensor([[10, 11, 12],
        [23, 24, 25]])


**Теория.** Здесь только правило формы из начала блока 6:

1. Номера осей: в `(4, 5, 6)` ось 0 = 4, ось 1 = 5, ось 2 = 6; `-1` = последняя.
2. `dim=k` → вычеркни ось k. `dim=(i, j)` → вычеркни обе.
3. `keepdim=True` → вместо вычёркивания поставь 1.

Пример на другой форме `(2, 7, 9)`: `sum(dim=0)` → `(7, 9)`; `mean(dim=2, keepdim=True)` → `(2, 7, 1)`; `sum(dim=(1, 2))` → `(2,)`.

**6.3** Предскажи формы **без запуска кода**: впиши кортежи руками, проверка сравнит с настоящими.

Дан `y` формы (4, 5, 6).

In [255]:
y = torch.rand(4, 5, 6)

In [256]:
shape_a = (4, 6)  # y.sum(dim=1)
shape_b = (1, 5, 6)  # y.mean(dim=0, keepdim=True)
shape_c = (5,)  # y.sum(dim=(0, 2))
shape_d = (4, 5)  # y.mean(dim=-1)

In [257]:
real = {
    "shape_a": y.sum(dim=1).shape,
    "shape_b": y.mean(dim=0, keepdim=True).shape,
    "shape_c": y.sum(dim=(0, 2)).shape,
    "shape_d": y.mean(dim=-1).shape,
}
mine = {"shape_a": shape_a, "shape_b": shape_b, "shape_c": shape_c, "shape_d": shape_d}
wrong = [k for k in real if tuple(real[k]) != mine[k]]
assert not wrong, "не совпало: " + ", ".join(
    f"{k}: ты {mine[k]}, на деле {tuple(real[k])}" for k in wrong
)
print("6.3 — верно")

6.3 — верно


## 7. CPU и GPU

У тебя PyTorch собран под CPU, поэтому `cuda` будет `False`, и это нормально. Важно выучить **шаблон**: код с ним работает и на CPU, и на GPU без правок.

**Теория.** `device` — где лежит тензор и кто с ним считает:

| `device` | что это |
|:---|:---|
| `"cpu"` | процессор, обычная память. По умолчанию |
| `"cuda"` / `"cuda:0"` | видеокарта NVIDIA (`:0` — номер карты, если их несколько) |

Зачем GPU: нейросеть — это огромное количество одинаковых умножений матриц, видеокарта делает тысячи таких операций одновременно → обучение в десятки раз быстрее.

Проверить доступность — функции `is_available()` в «ветках» `torch.cpu` и `torch.cuda`; возвращают `True` / `False`. Писать `"cuda"` напрямую нельзя — на машине без видеокарты код упадёт. Поэтому устройство выбирают условием (тернарный оператор Python):

```python
mode = "быстро" if 2 > 1 else "медленно"   # "быстро"
```

**7.1** Запиши, доступны ли CPU и GPU, в `cpu_ok` и `cuda_ok`. Затем заведи переменную `device`: строка `"cuda"`, если GPU доступен, иначе `"cpu"`.

In [258]:
cpu_ok = True if torch.cpu.is_available() else False
cuda_ok = True if torch.cuda.is_available() else False
device = "cuda" if torch.cuda.is_available() else "cpu"


print(cpu_ok, cuda_ok, device)

True False cpu


In [259]:
assert cpu_ok is True
assert cuda_ok == torch.cuda.is_available()
assert device == ("cuda" if cuda_ok else "cpu")
print("7.1 — верно")

7.1 — верно


**Теория.** Два способа положить тензор на нужное устройство:

```python
t = torch.tensor([5.0], device="cpu")   # сразу при создании
t = torch.tensor([5.0]).to("cpu")       # перенести уже созданный
t.device                                 # device(type='cpu') — где лежит сейчас
```

`.to(...)` **возвращает новый тензор**, а не меняет старый — результат надо присвоить: `t = t.to(device)`. Просто `t.to(device)` без присваивания ничего не перенесёт.

Правило: все тензоры в одной операции должны быть на **одном** устройстве — модель и данные переносят на `device` вместе.

**7.2** Создай `g` из `[1.0, 2.0]` с `requires_grad=True` и перенеси его на `device` через `.to(...)`.

In [260]:
g = torch.tensor([1.0, 2.0], requires_grad=True, device=device)

In [261]:
need_tensor(g, "g")
assert g.device.type == device, f"g на {g.device}, а должен быть на {device}"
print("7.2 — верно, g.device =", g.device)

7.2 — верно, g.device = cpu


**Теория.** `detach()` возвращает тензор с **теми же числами, но без истории операций**: `grad_fn` нет, `requires_grad=False`. Цепочка градиентов на нём обрывается — специально.

```python
k = torch.tensor([2.0], requires_grad=True) * 5
k            # tensor([10.], grad_fn=<MulBackward0>)
k.detach()   # tensor([10.])
```

Зачем: когда числа нужны **для вывода или сохранения** — график, `numpy`, запись в файл. Без `detach` к ним прицепится вся история (лишняя память), а `.numpy()` вообще не даст вызвать.

`.cpu()` — то же, что `.to("cpu")`. Методы пишут цепочкой, слева направо: сначала отцепить, потом перенести — `.detach().cpu()`.

**7.3** Посчитай `h = g * 3` — у `h` появится `grad_fn` (функция для градиентов). Получи `h_cpu` — копию `h` на CPU **без** связи с графом градиентов (`detach`, потом `cpu`).

In [262]:
h = g * 3
print(h)

h_cpu = h.detach()
print(h_cpu)

tensor([3., 6.], grad_fn=<MulBackward0>)
tensor([3., 6.])


In [263]:
need_tensor(h_cpu, "h_cpu")
assert h.grad_fn is not None
assert h_cpu.grad_fn is None and not h_cpu.requires_grad, "нужен detach()"
assert h_cpu.device.type == "cpu"
print("7.3 — верно")

7.3 — верно


> Два уточнения к видео:
> - тензоры на разных устройствах складывать нельзя — будет `RuntimeError: Expected all tensors to be on the same device`. Проверить у себя не выйдет без GPU, просто запомни;
> - копия создаётся, только когда тензор реально переезжает. Если он уже на нужном устройстве, `.to(...)` возвращает **тот же** тензор: проверь `g.to("cpu") is g` на CPU.

## 8. Разминка на формы

В видео этого почти нет, но в нейросетях это главное. Формы пиши в комментариях рядом с кодом: `# (B, C, H, W)`.

**Теория.** Как хранится батч картинок `(B, C, H, W)` — матрёшка: батч → картинки → каналы → матрица пикселей.

```python
x = torch.arange(24).view(2, 3, 2, 2)   # 2 картинки, 3 канала, 2×2 пикселя
x[0]            # картинка 0 — (3, 2, 2)
x[0, 1]         # её канал 1 (G) — матрица 2×2: [[4, 5], [6, 7]]
x[0, 1, 0, 1]   # один пиксель — tensor(5)
```

В выводе: одна пустая строка — граница между каналами, две — между картинками.

Модели в PyTorch **всегда** принимают батч: первая ось — номер примера. Даже для одного примера нужна ось батча размера 1.

#### Теория: формы в рекомендательных системах

В рексисе картинки редкость — основные данные таблицы и истории действий пользователей. Правила те же, меняются только смыслы осей:

| форма | что это |
|:---|:---|
| `(B,)` | батч ID пользователей или товаров (целые числа) |
| `(B, F)` | признаки: возраст, цена, категория… |
| `(B, E)` | эмбеддинги — вектор длины E, описывающий пользователя или товар |
| `(B, T)` | история: ID последних T товаров каждого пользователя |
| `(B, T, E)` | та же история, но каждый товар заменён своим эмбеддингом |

Та же матрёшка: батч → пользователи → товары истории → (если есть) вектор товара:

```python
hist = torch.tensor([[11, 42, 7],
                     [ 5,  5, 90]])   # (B, T) = 2 пользователя, по 3 последних товара
hist[0]       # tensor([11, 42,  7])  — история пользователя 0, форма (3,)
hist[0, -1]   # tensor(7)            — последний товар, который он смотрел
hist[:, 0]    # tensor([11,  5])     — первый товар истории у каждого пользователя

# если каждый ID заменить вектором длины 4 — появится ось E:
# (2, 3) → (2, 3, 4) = (B, T, E)
```

Правило про батч то же: модель ждёт ось `B` первой. Нужно сделать рекомендацию **одному** пользователю — его историю `(T,)` превращают в «батч из одного» `(1, T)`.

**8.1** Рекомендация одному пользователю. Даны:
- `history` — ID 5 последних товаров, которые он смотрел, форма `(T,)` = (5,);
- `history_emb` — эмбеддинги этих же товаров, форма `(T, E)` = (5, 4).

Модель принимает только батч. Получи:
- `history_batch` формы `(1, 5)`;
- `history_emb_batch` формы `(1, 5, 4)`.

In [264]:
history = torch.tensor(
    [101, 7, 42, 7, 15]
)  # (T,) — ID последних 5 товаров (товар 7 смотрели дважды)
history_emb = torch.rand(5, 4)  # (T, E) — у каждого из 5 товаров вектор длины 4

In [265]:
history_batch = history.unsqueeze(0)
# history_emb_batch = history_emb.unsqueeze(0)
# history_emb_batch = history_emb.view(1, 5, 4)
history_emb_batch = history_emb.reshape(1, 5, 4)

In [266]:
need_tensor(history_batch, "history_batch")
need_tensor(history_emb_batch, "history_emb_batch")
assert history_batch.shape == (
    1,
    5,
), f"форма {tuple(history_batch.shape)}, а нужна (1, 5)"
assert history_emb_batch.shape == (
    1,
    5,
    4,
), f"форма {tuple(history_emb_batch.shape)}, а нужна (1, 5, 4)"
assert torch.equal(
    history_batch[0], history
), "внутри батча должна лежать та же история — числа не меняются"
assert torch.equal(history_emb_batch[0], history_emb)
print("8.1 — верно")

8.1 — верно


**Теория.** Оба принимают **список** тензоров:

| | `torch.stack` | `torch.cat` |
|:---|:---|:---|
| ось | создаёт **новую** | склеивает по **существующей** |
| формы на входе | строго одинаковые | могут отличаться по оси склейки |
| два `(3,)` | → `(2, 3)` | → `(6,)` |

```python
p1 = torch.tensor([7, 8, 1])   # (3,)
p2 = torch.tensor([0, 9, 4])   # (3,)

torch.stack([p1, p2])          # [[7, 8, 1],
                               #  [0, 9, 4]]       (2, 3) — новая ось 0: «какой тензор по счёту»
                               #  каждая СТРОКА — один тензор

torch.stack([p1, p2], dim=1)   # [[7, 0],
                               #  [8, 9],
                               #  [1, 4]]          (3, 2) — новая ось в конце
                               #  каждый СТОЛБЕЦ — один тензор

torch.cat([p1, p2])            # [7, 8, 1, 0, 9, 4]  (6,) — просто приклеили
```

Где встречается: `DataLoader` собирает батч из отдельных примеров именно через `stack`: 32 картинки `(3, 28, 28)` → `(32, 3, 28, 28)`.

**8.2** Два способа собрать тензоры:
- `torch.stack` — создаёт **новую** ось;
- `torch.cat` — склеивает по **существующей**.

Даны `u1` и `u2` формы (3,). Получи `st` формы (2, 3) и `ct` формы (6,).

In [267]:
u1 = torch.tensor([1, 2, 3])
u2 = torch.tensor([4, 5, 6])

In [268]:
st = torch.stack([u1, u2], dim=0)
ct = torch.cat([u1, u2])

In [269]:
need_tensor(st, "st")
need_tensor(ct, "ct")


assert st.shape == (2, 3) and st.tolist() == [[1, 2, 3], [4, 5, 6]]


assert ct.shape == (6,) and ct.tolist() == [1, 2, 3, 4, 5, 6]


print("8.2 — верно")

8.2 — верно


**Теория.** `permute(...)` — **переставляет оси местами**. В скобках — номера **старых** осей в новом порядке: «новая ось 0 = старая ось такая-то, новая ось 1 = ...».

```python
t = torch.zeros(2, 3, 4)
t.permute(1, 0, 2).shape   # (3, 2, 4): новая 0 = старая 1, новая 1 = старая 0, новая 2 = старая 2
t.permute(2, 0, 1).shape   # (4, 2, 3)
```

Не путать с `reshape`: `reshape(3, 2, 4)` даст ту же форму, но **перераскладывает числа подряд** и перемешает смысл (пиксели разъедутся по каналам). `permute` сохраняет, какое число к какому каналу и пикселю относится.

Для 2D `permute(1, 0)` — то же, что `.T`.

**8.3** `permute` меняет порядок осей. Картинки часто хранят как (H, W, C), а PyTorch ждёт (C, H, W). Получи `chw` из `hwc`.

In [270]:
hwc = torch.rand(4, 5, 3)  # (H, W, C)

In [271]:
chw = hwc.permute(2,0,1)

In [272]:
need_tensor(chw, "chw")
assert chw.shape == (3, 4, 5)
assert torch.equal(chw[0], hwc[:, :, 0]), "первый канал должен совпасть с hwc[:, :, 0]"
print("8.3 — верно")

8.3 — верно


## 9. Итоговая задача: батч историй пользователей (рексис)

**Теория.** Почти всё — из блоков выше, плюс одна новая функция `argmax`:

| шаг | инструмент | где был |
|:---|:---|:---|
| среднее по оси | `mean(dim=...)` | блок 6 |
| форма для вычитания | `keepdim=True` + broadcasting | 6.2 |
| вытянуть каждую историю в вектор | `view(B, -1)`: ось батча оставить, остальное — в одну | 4.1 |
| поэлементное умножение + сумма по последней оси | `*`, `sum(dim=-1)` | 5.1, блок 6 |
| номер самого большого элемента | `argmax()` — **новое** | ниже |
| число из тензора | `.item()` | 2.2 |

#### Эмбеддинг пользователя из истории

Простейший способ получить вектор пользователя — **усреднить эмбеддинги товаров**, которые он смотрел. У `(B, T, E)` «усреднить историю каждого пользователя» = свернуть ось `T`; останется `(B, E)` — по вектору на пользователя.

#### Похожесть пользователя и товара — скалярное произведение

Перемножить векторы покоординатно и сложить. Чем больше число (скор), тем товар ближе ко вкусу пользователя. Это основа почти любой рекомендательной модели.

```python
u = torch.tensor([1.0, 0.0, 2.0])   # пользователь
i = torch.tensor([0.5, 3.0, 1.0])   # товар
(u * i).sum()                       # 1·0.5 + 0·3 + 2·1 = tensor(2.5)
```

Для батча пользователей `(B, E)` и одного товара `(E,)` broadcasting растянет товар на всех: `(B, E) * (E,)` → `(B, E)`, потом сумма по последней оси → `(B,)` — по скору на пользователя.

#### `argmax()` — номер самого большого элемента

```python
torch.tensor([0.2, 0.9, 0.4]).argmax()   # tensor(1) — самое большое (0.9) стоит на месте 1
```

Возвращает **номер**, а не само значение (значение — это `max()`). В рекомендациях так выбирают «лучший» вариант: самый подходящий товар или пользователя.

#### Задача

`H` — батч историй: 8 пользователей, у каждого 5 последних товаров, у каждого товара эмбеддинг длины 4 → `(B, T, E)` = (8, 5, 4). `item` — эмбеддинг нового товара-кандидата, `(E,)` = (4,).

1. `user_emb` — эмбеддинг каждого пользователя = среднее эмбеддингов его истории, форма (8, 4).
2. `user_emb_k` — то же, но с `keepdim=True`, форма (8, 1, 4).
3. `H_centered` — из каждого товара истории вычти эмбеддинг **его** пользователя (у каждого пользователя среднее по истории станет 0), форма (8, 5, 4).
4. `H_flat` — историю каждого пользователя из `H_centered` вытяни в один вектор, форма (8, 20). Так историю можно подать в `nn.Linear` — это будет в следующих видео.
5. `scores` — похожесть каждого пользователя (`user_emb`) на `item`, форма (8,).
6. `best_user` — номер пользователя, которому `item` подходит больше всего, как обычное число Python.

In [273]:
torch.manual_seed(1)
H = torch.rand(
    8, 5, 4
)  # (B, T, E) — 8 пользователей × 5 товаров истории × эмбеддинг длины 4
item = torch.rand(4)  # (E,) — эмбеддинг товара-кандидата
print("🔴 ~ item:", item)


🔴 ~ item: tensor([0.1264, 0.6924, 0.6601, 0.8238])


In [282]:
user_emb = H.mean(dim=1)
user_emb_k = H.mean(dim=1, keepdim=True)
H_centered = H - user_emb_k
H_flat = H_centered.view(8,20)
scores = (user_emb * item).sum(dim=1)
best_user = scores.argmax().item()


In [275]:
for t, name in [
    (user_emb, "user_emb"),
    (user_emb_k, "user_emb_k"),
    (H_centered, "H_centered"),
    (H_flat, "H_flat"),
    (scores, "scores"),
]:
    need_tensor(t, name)

assert user_emb.shape == (
    8,
    4,
), f"форма {tuple(user_emb.shape)}, а нужна (8, 4): сворачиваем ось T"
assert torch.allclose(
    user_emb, H.mean(dim=1)
), "user_emb — среднее по истории каждого пользователя"
assert user_emb_k.shape == (
    8,
    1,
    4,
), f"форма {tuple(user_emb_k.shape)}, а нужна (8, 1, 4)"
assert H_centered.shape == H.shape
assert torch.allclose(
    H_centered.mean(dim=1), torch.zeros(8, 4), atol=1e-6
), "у каждого пользователя среднее по истории должно стать 0"
assert H_flat.shape == (8, 20), f"форма {tuple(H_flat.shape)}, а нужна (8, 20)"
assert torch.equal(H_flat[0], H_centered[0].reshape(-1)), "вытягивать нужно H_centered"
assert scores.shape == (
    8,
), f"форма {tuple(scores.shape)}, а нужна (8,) — по скору на пользователя"
assert torch.allclose(
    scores, user_emb @ item
), "скор = скалярное произведение user_emb и item (см. теорию)"
assert isinstance(
    best_user, int
), f"best_user — {type(best_user).__name__}, а нужно обычное число (int)"
assert best_user == int(scores.argmax()), "это не пользователь с самым большим скором"
print("9 — верно. Блок пройден. Лучший пользователь для item:", best_user)

9 — верно. Блок пройден. Лучший пользователь для item: 3


## Что запомнить из видео

| # | Правило |
|:---|:---|
| 1 | `torch.tensor(список, dtype=..., requires_grad=..., device=...)` |
| 2 | `requires_grad=True` — только для float |
| 3 | `shape` == `size()`, `ndim` — число осей |
| 4 | индексация даёт тензор, число — через `item()` (например, для loss) |
| 5 | `view` / `reshape` меняют форму; `-1` — посчитай сам; если сомневаешься — `reshape` |
| 6 | `unsqueeze` добавляет ось 1, `squeeze` убирает |
| 7 | `dim` — ось, которая исчезает; `keepdim=True` оставляет её с размером 1 |
| 8 | `device = "cuda" if torch.cuda.is_available() else "cpu"`, потом `.to(device)` |
| 9 | тензоры для одной операции — на одном устройстве |
| 10 | `detach()` — отцепить от графа градиентов |